In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
data = pd.read_csv('../data/processed/cleaned_break_data.csv')
data.sample(5)

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
from datetime import datetime, date
data['INCIDENT_DATE'] = pd.to_datetime(data['INCIDENT_DATE']).dt.date

In [ ]:
data.sample(5)

Possible columns to drop still:
- Global ID
- Units Impacted
- Hours Impacted
- possibly more after further analysis...


In [ ]:
data['HOUR_IMPACTED'].value_counts()

In [ ]:
data['UNITS_IMPACTED'].value_counts()

Can't really see these variables being too valuable for any sort of model so we can go ahead and remove them along with Global ID

In [ ]:
data.drop(['GLOBALID', 'HOUR_IMPACTED', 'UNITS_IMPACTED'], axis=1, inplace=True)

In [ ]:
data.isna().sum() # just double checking even though we already cleaned the data in our other notebook

In [ ]:
data['ASSET_MATERIAL'].value_counts()

In [ ]:
data['ASSETID'].nunique()

In [ ]:
num_break = {}

for pipe in data['ASSETID']:
    if pipe in num_break:
        num_break[pipe] += 1
    else:
        num_break[pipe] = 1

In [ ]:
data['NUM_BREAKS'] = data['ASSETID'].map(num_break)

In [ ]:
data.tail()

In [ ]:
data.shape

In [ ]:
data.NUM_BREAKS.sum()

In [ ]:
data['ASSET_YEAR_INSTALLED'] = pd.to_datetime(data['ASSET_YEAR_INSTALLED'], format='%Y')

In [ ]:
data.sample(5)

In [ ]:
data['ASSET_YEAR_INSTALLED'].min()

In [ ]:
data['ASSET_YEAR_INSTALLED'].max()

In [ ]:
data['INCIDENT_DATE'].min()

In [ ]:
data['AGE'] = (np.floor((pd.to_datetime(data['INCIDENT_DATE']) - 
                        pd.to_datetime(data['ASSET_YEAR_INSTALLED'])).dt.days / 365.25)).astype(int)

# I found this code on stack overflow to help calculate age

# df['age'] = (np.floor((pd.to_datetime(df['dte']) - 
#              pd.to_datetime(dob)).dt.days / 365.25)).astype(int)

In [ ]:
data.sample(5)

In [ ]:
data.info()

In [ ]:
data.columns = data.columns.str.lower()

In [ ]:
data['status'].value_counts()

In [ ]:
data.drop('status', axis=1, inplace=True)

In [ ]:
data['break_type'].value_counts()

In [ ]:
data['break_nature'].value_counts()

In [ ]:
data['break_apparent_cause'].value_counts()

In [ ]:
data.sample(5)

We can see that we've got 7 columns where there are what we can call "binary" values - Y (yes) & N (no). We're going to one-hot encode these variables so we can start to put the data into proper modeling format but first, we'll double check to make sure that there are only 2 values for each column.

In [ ]:
binary_cols = data[['positive_pressure_maintaned', 'air_gap_maintaned', 'mechanical_removal', 
                    'flushing_excavation', 'higher_velocity_flushing', 'anode_installed', 'asset_exists']]
for col in binary_cols:
    print(binary_cols[col].value_counts())

In [ ]:
binary_cols = ['positive_pressure_maintaned', 'air_gap_maintaned', 'mechanical_removal', 
                    'flushing_excavation', 'higher_velocity_flushing', 'anode_installed', 'asset_exists']

data = pd.get_dummies(data, columns=binary_cols, drop_first=True)

In [ ]:
data.sample(5)

In [ ]:
binary_cols = data[['positive_pressure_maintaned_Y', 'air_gap_maintaned_Y', 'mechanical_removal_Y', 
                    'flushing_excavation_Y', 'higher_velocity_flushing_Y', 'anode_installed_Y', 'asset_exists_Y']]
for col in binary_cols:
    print(binary_cols[col].value_counts())

In [ ]:
data = data.rename(columns={'positive_pressure_maintaned_Y': 'positive_pressure_maintaned',
                            'air_gap_maintaned_Y': 'air_gap_maintaned',
                            'mechanical_removal_Y': 'mechanical_removal',
                            'flushing_excavation_Y': 'flushing_excavation',
                            'higher_velocity_flushing_Y': 'higher_velocity_flushing',
                            'anode_installed_Y': 'anode_installed',
                            'asset_exists_Y': 'asset_exists'})

Columns to transform still:
- break type
- break nature
- break apparent cause
- break categorization
- asset material
- maybe think about dropping the street name since that would only be for EDA use

In [ ]:
cat_cols = data[['break_type', 'break_nature', 'break_apparent_cause',
                 'break_categorization', 'asset_material']]

for col in cat_cols:
    print(cat_cols[col].value_counts(), "\n" + "-"*55)

I'll break down what I'm thinking for how I'm going to encode these columns:

Break type is easy and I'll just one hot encode it or use `get_dummies`. Break nature is going to be a little trickier since it has high cardinality. I think I'm going to combine the features that have matching break nature as the first name for the feature. For example - `CIRCUMFERENTIAL AND FITTING/JOINT` will be added to `CIRCUMFERENTIAL`. `CORROSION` has a lot of subcategories and so all of those will be joined into the original `CORROSION` column. Lastly `FITTING/JOINT AND LONGITUDINAL` will be added to `FITTING/JOINT`.

For break apparent cause I'll add the `UNKNOWN` values into `OTHER` to be consistent since `OTHER` is the majority value.

Break categorization is relatively easy as well and I can OH encode them or use `get_dummies` again.

I'm currently doing some digging on the different asset materials to see what can be done for these values. I'm already doubting I can combine any since they represent unique pipe materials but maybe research will help turn something up.
- CI = Cast Iron
- DI = Ductile Iron
- PVC = Polyvinyl Chloride
- PVCO = Molecularly Oriented PVC
- CPP = Concrete Pressure Pipe
- XXX = this one is still a mystery
- HDPE = High-density Polyethylene
- AC = Asbestos Cement
- PVCB = not exactly sure what this could be. A google search turns up only these types of PVC pipes:
    - Unplasticized PVC (PVC-U)
    - Chlorinated PVC (C-PVC)
    - Molecularly oriented (PVC-O)
    - High-impact PVC (PVC-Hi)
- COP = Copper
- PE = polyethylene

In [ ]:
data['break_nature'] = data['break_nature'].replace(['CIRCUMFERENTIAL AND FITTING/JOINT'], 'CIRCUMFERENTIAL')
data['break_nature'] = data['break_nature'].replace(['CORROSION AND CIRCUMFERENTIAL',
                                                         'CORROSION AND LONGITUDINAL',
                                                         'CORROSION AND FITTING/JOINT',
                                                         'CORROSION - ROBAR SADDLE CORRODED AT SEAM'], 'CORROSION')
data['break_nature'] = data['break_nature'].replace(['FITTING/JOINT AND LONGITUDINAL'], 'FITTING/JOINT')

In [ ]:
data.break_nature.value_counts()

In [ ]:
data['break_apparent_cause'] = data['break_apparent_cause'].replace(['UNKNOWN'], 'OTHER')

In [ ]:
data.break_apparent_cause.value_counts()

In [ ]:
data['age'].describe()

In [ ]:
num_neg = [age for age in data['age'] if age < 0]

In [ ]:
len(num_neg)

I wanted to see what the range for the negative ages were since there are a decent amount of negative ages. The possibility of ages to be negative is from the replacement of the pipes since their last break. For instance, the one pipe had a break in 2001 but was replaced in 2006 so it has an age of -5.

Technically what I could do is remove all instances where the asset does not exist since we're only insterested in predicting the failure of an existing pipe.

In [ ]:
data.roadsegmentid.value_counts()

In [ ]:
data['roadsegmentid'].value_counts().describe()

In [ ]:
data['incident_date'] = pd.to_datetime(data['incident_date'])

In [ ]:
data['asset_year_installed'] = pd.to_datetime(data['asset_year_installed'])

In [ ]:
data['incident_date'].dt.year.value_counts()

In [ ]:
data['asset_exists'].value_counts()

In [ ]:
plt.figure(figsize=(12, 10))
data['incident_date'].dt.year.value_counts().sort_index().plot(kind='bar')
plt.show();

In [ ]:
breaks_per_year = pd.Series(data['incident_date'].dt.year.value_counts().sort_index())
breaks_per_year.index

In [ ]:
plt.figure(figsize=(15, 10))
sns.barplot(x=breaks_per_year.index, y=breaks_per_year.values)
plt.xticks(rotation=45)
plt.show();

Let's drop observations where the asset doesn't exist.

In [ ]:
# shape before dropping
print(data.shape)
data = data[data['asset_exists'] == 1]
# shape after dropping
print(data.shape)

In [ ]:
data.head()

In [ ]:
# drop watbreakincidentid
data.drop('watbreakincidentid', axis=1, inplace=True)

In [ ]:
data.head()

Let's encode our categorical variables and start to save the data as a separate set so we keep some originality for the EDA.

Categories to encode:
- `break_type`
- `break_nature`
- `break_apparent_cause`
- `break_categorization`
- `asset_material`

In [ ]:
data['objectid'].nunique()

In [ ]:
data_copy = data.copy()
data_copy.drop(['objectid', 'street', 'assetid'], axis=1, inplace=True)

In [ ]:
# make sure there aren't any negative ages anymore
data_copy['age'].describe()

In [ ]:
plt.figure(figsize=(12, 8))
sns.histplot(data_copy['age'], kde=True)
plt.show();

I'll visualize all of the unique values for each category that needs to be encoded so I know how many different values are in each column. I could get dummy variables for the categories but that would increase the feature space more than I would like. I thought about just creating a dictionary of each column with corresponding values and mapping them to the column but this could also be time consuming. Let's try and use pandas to our advantage here and convert each column to a category and call `cat.codes` on them, this is known as label encoding.

In [ ]:
cat_features = data_copy[['break_type', 'break_nature', 'break_apparent_cause', 'break_categorization', 'asset_material']]

for feature in cat_features:
    print(data_copy[feature].value_counts())
    print('-'*30)

In [ ]:
for feature in data_copy[['break_type', 'break_nature', 'break_apparent_cause', 'break_categorization', 'asset_material']]:
    data_copy[feature] = data_copy[feature].astype('category')
    data_copy[feature] = data_copy[feature].cat.codes


In [ ]:
data_copy.head()

In [ ]:
cat_features = data_copy[['break_type', 'break_nature', 'break_apparent_cause', 'break_categorization', 'asset_material']]

for feature in cat_features:
    print(data_copy[feature].value_counts())
    print('-'*30)

In [ ]:
data_copy.drop('roadsegmentid', axis=1, inplace=True)

In [ ]:
data_copy.head()

In [ ]:
data_copy['asset_size'].value_counts()

In [ ]:
data_copy.rename(columns={'age': 'age_at_break'}, inplace=True)

In [ ]:
plt.figure(figsize=(22, 12))
sns.barplot(data=data_copy, x='age_at_break', y='num_breaks')
plt.xticks(rotation=90)
plt.show();

In [ ]:
data_copy['num_breaks'].describe()

In [ ]:
data_copy['age_at_break'].describe()

In [ ]:
max_age = data_copy['age_at_break'].max()
num_breaks_max_age = data_copy['num_breaks'].loc[data_copy['age_at_break'] == max_age]

max_age_failure_rate = num_breaks_max_age / max_age

print(max_age_failure_rate)

In [ ]:
# how many values have age_at_break = 0
data_copy['age_at_break'].loc[data_copy['age_at_break'] == 0].count()

In [ ]:
# calculate the failure rate from num_breaks and age
data_copy['failure_rate'] = round(data_copy['num_breaks'] / data_copy['age_at_break'], 2)
data_copy.sample(5)

In [ ]:
data_copy['age_at_break'].describe()

In [ ]:
data_copy['failure_rate'].describe()

In [ ]:
# display failure rates that are inf
data_copy['failure_rate'].loc[data_copy['failure_rate'] == np.inf]

In [ ]:
# print failure rates that are inf
infinity_failure_rates = data_copy[data_copy['failure_rate']== np.inf]
infinity_failure_rates

Manually calculating the age for the dates above that are `inf` values. The age will be represented as a decimal number representing days. For example, the top row where date of incident is 2011-12-15 and asset year installed is 2011-01-01 would have an age of 0.348 = 348 days old.

- index 643 = 0.348
- index 1144 = 0.183
- index 1300 = 0.345
- index 1383 = 0.186
- index 1439 = 0.348
- index 1774 = 0.204

In [ ]:
# drop rows with inf failure rates
data_copy.drop(infinity_failure_rates.index, axis=0, inplace=True)

In [ ]:
data_copy.loc[data_copy['failure_rate'] == np.inf]

In [ ]:
data_copy.describe()

In [ ]:
data_copy.info()

Now that that's fixed, I'm going to recalculate the failure rates to see if the `np.inf` values have been talen care of.

In [ ]:
data_copy['failure_rate'] = round(data_copy['num_breaks'] / data_copy['age_at_break'], 4)

In [ ]:
data_copy.describe()

In [ ]:
data_copy.head()

In [ ]:
model_data = data_copy.copy()
model_data.drop(['longitude', 'latitude', 'incident_date', 'asset_year_installed'], axis=1, inplace=True)
model_data.to_csv('../data/processed/model_data.csv', index=False)

In [ ]:
# data_copy.to_csv('../data/processed/final_data.csv', index=False)